In [2]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from pathlib import Path

# Dataset mới sau khi xử lý và xoá trùng
DATA_PATH = '../../Rain-Forecast/datasets/processed/email_dataset_github_processed.csv'
df = pd.read_csv(DATA_PATH)

print('Dataset đã xử lý:')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print()

# 1. Kiểm tra cân bằng lớp
if 'isSpam' in df.columns:
    class_counts = df['isSpam'].value_counts()
    print('Số lượng mỗi lớp:')
    print(class_counts)
    print('Tỉ lệ mỗi lớp:')
    print(class_counts / len(df))
else:
    print('Không tìm thấy cột isSpam để kiểm tra cân bằng lớp.')

print()

# 2. Kiểm tra độ dài email
if 'msg' in df.columns:
    df['num_words'] = df['msg'].apply(lambda x: len(str(x).split()))
    df['num_chars'] = df['msg'].apply(lambda x: len(str(x)))
    print('Số từ trên mỗi email:')
    print('Min:', df['num_words'].min())
    print('Max:', df['num_words'].max())
    print('Mean:', df['num_words'].mean())
    print()
    print('Số ký tự trên mỗi email:')
    print('Min:', df['num_chars'].min())
    print('Max:', df['num_chars'].max())
    print('Mean:', df['num_chars'].mean())
else:
    print('Không tìm thấy cột msg để kiểm tra số từ.')

print()

# 3. Kiểm tra dữ liệu bẩn
print('Kiểm tra dữ liệu bẩn:')
if 'msg' in df.columns:
    empty_msgs = df['msg'].isnull().sum() + (df['msg'].astype(str).str.strip() == '').sum()
    print(f'Số tin nhắn rỗng: {empty_msgs}')
    special_char_msgs = df['msg'].apply(lambda x: bool(re.fullmatch(r'[^\w\s]+', str(x))))
    print(f'Số tin nhắn chỉ chứa ký tự đặc biệt: {special_char_msgs.sum()}')
else:
    print('Không tìm thấy cột msg để kiểm tra dữ liệu bẩn.')

print()

# 4. Kiểm tra trùng nội dung và label sau khi đã xử lý
if {'msg', 'isSpam'}.issubset(df.columns):
    dup_msg_count = df.duplicated(subset=['msg']).sum()
    dup_pair_count = df.duplicated(subset=['msg', 'isSpam']).sum()
    conflict_msgs = df.groupby('msg')['isSpam'].nunique()
    conflict_msgs = conflict_msgs[conflict_msgs > 1].index.tolist()

    print('Kiểm tra trùng lặp:')
    print(f'Số dòng trùng theo msg: {dup_msg_count}')
    print(f'Số dòng trùng hoàn toàn theo msg + isSpam: {dup_pair_count}')
    if conflict_msgs:
        print('CẢNH BÁO: Có nội dung giống nhau nhưng label khác nhau:')
        conflict_view = df[df['msg'].isin(conflict_msgs)][['msg', 'isSpam']].drop_duplicates().sort_values('msg')
        print(conflict_view.to_string(index=False))
    else:
        print('Không phát hiện nội dung giống nhau nhưng label khác nhau.')
else:
    print('Không đủ cột msg/isSpam để kiểm tra trùng.')

print()

# 5. Xem nhanh vài dòng đầu
print('5 mẫu đầu của dataset đã xử lý:')
print(df[['msg', 'isSpam']].head().to_string(index=False))

Dataset đã xử lý:
Shape: (273722, 3)
Columns: ['msg', 'isSpam', 'num_words']

Số lượng mỗi lớp:
isSpam
1    206218
0     67504
Name: count, dtype: int64
Tỉ lệ mỗi lớp:
isSpam
1    0.753385
0    0.246615
Name: count, dtype: float64

Số từ trên mỗi email:
Min: 2
Max: 18
Mean: 11.499397198617576

Số ký tự trên mỗi email:
Min: 14
Max: 112
Mean: 81.13623311242793

Kiểm tra dữ liệu bẩn:
Số tin nhắn rỗng: 0
Số tin nhắn chỉ chứa ký tự đặc biệt: 0

Kiểm tra trùng lặp:
Số dòng trùng theo msg: 0
Số dòng trùng hoàn toàn theo msg + isSpam: 0
Không phát hiện nội dung giống nhau nhưng label khác nhau.

5 mẫu đầu của dataset đã xử lý:
                                                                                         msg  isSpam
                             The book club meeting is moved to 16 Apr 2025 at Sarah's place.       0
    Exclusive Forex trading signal! 95% win rate. Join free: http://verify-secure.com/nzin25       1
                     Increase height naturally in 30 days! Rs 16,500 o